# Class 13 · OOP in Practice, and Regular Expressions

Python · AIML Jul26 Batch 3

Every example from the class, in order. Run a cell with **Shift+Enter**. Cells that use `input()` wait for you to type an answer below them.

- Class page: https://prateekscaler.github.io/AIML-Jul26-Batch3-Python/Class-13-OOP-Practice-And-Regex/
- Notes, assignment and interview questions: https://github.com/prateekScaler/AIML-Jul26-Batch3-Python/tree/main/Class-13-OOP-Practice-And-Regex


## Encapsulation

`01_encapsulation.py`

Encapsulation: keep the data inside, change it only through methods.
The open version: any line can set balance to anything.


In [ ]:
class OpenAccount:
    def __init__(self, balance):
        self.balance = balance

account = OpenAccount(100)
account.balance = -500                # nothing stops it
print(account.balance)                # -500

# The fix, part 1: two underscores, no methods yet.
class BankAccount:
    def __init__(self, balance):
        self.__balance = balance

account = BankAccount(100)
# print(account.__balance)            # AttributeError: no attribute '__balance'

# The fix, part 2: the methods are the only door.
class BankAccount:
    def __init__(self, balance):
        self.__balance = balance      # two underscores: internal

    def get_balance(self):
        return self.__balance

    def deposit(self, amount):
        if amount > 0:
            self.__balance += amount

account = BankAccount(100)
account.deposit(50)
account.deposit(-500)                 # refused quietly by the rule inside
print(account.get_balance())          # 150

# print(account.__balance)            # AttributeError: no attribute '__balance'
print(account._BankAccount__balance)  # name mangling: a speed bump, not a lock


## Inheritance and polymorphism

`02_inheritance_and_polymorphism.py`

Inheritance: a class reuses another class's code.


In [ ]:
class Employee:
    def __init__(self, name, salary):
        self.name = name
        self.salary = salary

    def describe(self):
        return f"{self.name} is an employee."

class Manager(Employee):
    def __init__(self, name, salary, team_size):
        super().__init__(name, salary)     # let Employee set up name and salary
        self.team_size = team_size

    def describe(self):                    # overriding: this version wins for a Manager
        return f"{self.name} manages a team of {self.team_size}."

class Developer(Employee):
    def __init__(self, name, salary, language):
        super().__init__(name, salary)
        self.language = language

    def describe(self):
        return f"{self.name} writes code in {self.language}."

emp = Employee("Prajnika", 60000)
mgr = Manager("Chitransh", 90000, 5)
dev = Developer("Rahul", 75000, "Python")

print(emp.describe())
print(mgr.describe())
print(mgr.team_size)

# Polymorphism: one call, many forms.
for person in [mgr, dev, emp]:
    print(person.describe())


## Inheritance scenarios

`03_inheritance_scenarios.py`

Six inheritance scenarios: a chain of parents, extending a method, isinstance,
a child's own method, a child that adds a check, and is-a versus has-a.


In [ ]:
class Employee:
    def __init__(self, name, salary):
        self.name = name
        self.salary = salary

    def describe(self):
        return f"{self.name} is an employee."


class Manager(Employee):
    def __init__(self, name, salary, team_size):
        super().__init__(name, salary)
        self.team_size = team_size

    def describe(self):
        return f"{self.name} manages a team of {self.team_size}."

    def approve_leave(self, who):
        return f"{self.name} approved leave for {who}."


# 1. A chain: Director is a Manager, and a Manager is an Employee
class Director(Manager):
    def __init__(self, name, salary, team_size, region):
        super().__init__(name, salary, team_size)
        self.region = region

    # 2. Extend, do not replace: the parent's sentence, plus one more
    def describe(self):
        return super().describe() + f" Region: {self.region}."


d = Director("Sangeeta", 150000, 40, "South")
print(d.describe())
print(d.salary)                       # set by Employee.__init__, two levels up

# 3. isinstance: a Director is a Manager and an Employee; an Employee is not a Manager
e = Employee("Prajnika", 60000)
print(isinstance(d, Employee), isinstance(e, Manager))
print(type(d) == Manager, type(d).__name__)

# 4. A child's own method is not on the parent
print(d.approve_leave("Rahul"))
try:
    e.approve_leave("Rahul")
except AttributeError as err:
    print("AttributeError:", err)

# 5. A child adds a check, then hands over to the parent
class Wallet:
    def __init__(self, balance):
        self.balance = balance

    def pay(self, amount):
        self.balance -= amount
        print(f"Paid {amount}. Balance {self.balance}")


class StudentWallet(Wallet):
    def pay(self, amount):
        if amount > 2000:
            print("Student wallets are limited to 2000 per payment.")
            return
        super().pay(amount)


w = StudentWallet(5000)
w.pay(3000)
w.pay(1500)

# 6. is-a versus has-a: a Transaction is not a kind of Account, so no inheritance
class Transaction:
    def __init__(self, payee, amount):
        self.payee = payee
        self.amount = amount


class Account:
    def __init__(self, owner):
        self.owner = owner
        self.history = []          # has-a: an Account has Transactions

    def pay(self, payee, amount):
        self.history.append(Transaction(payee, amount))


a = Account("Vinay")
a.pay("Sumit", 600)
print(len(a.history), a.history[0].payee)


## Employees

`04_employees.py`

Mini project: every idea of the class in one program.


In [ ]:
class Employee:
    def __init__(self, name, salary):
        self.name = name
        self.__salary = salary

    def get_salary(self):
        return self.__salary

    def describe(self):
        return f"{self.name} is an employee."


class Manager(Employee):
    def __init__(self, name, salary, team_size):
        super().__init__(name, salary)
        self.team_size = team_size

    def describe(self):
        return f"{self.name} manages a team of {self.team_size}."


class Developer(Employee):
    def __init__(self, name, salary, language):
        super().__init__(name, salary)
        self.language = language

    def describe(self):
        return f"{self.name} writes code in {self.language}."


employees = [
    Manager("Chitransh", 90000, 5),
    Developer("Rahul", 75000, "Python"),
    Employee("Prajnika", 60000),
]

for emp in employees:
    print(emp.describe())
    print("Salary:", emp.get_salary())


## Regex basics

`05_regex_basics.py`

Regular expressions: describe the shape of text, not the exact text.
search finds a pattern anywhere; match only at the start; fullmatch needs the whole string.


In [ ]:
import re

text = "concatenate"
print(re.search(r"cat", text))      # <re.Match object; span=(3, 6), match='cat'>
print(re.match(r"cat", text))       # None: the string starts with "con", not "cat"

# . is one blank to fill: exactly one character, any character
for word in ["cat", "cot", "ct", "coat"]:
    print(word, bool(re.search(r"c.t", word)))

# ^ is the starting line, $ is the finish line
messages = ["UPI paid", "Paid by UPI"]
for m in messages:
    print(m, bool(re.search(r"^UPI", m)))
for m in messages:
    print(m, bool(re.search(r"UPI$", m)))

# \ takes away the special job: \. is a real dot
name = "bill-pdf"
print(bool(re.search(r"bill.pdf", name)))   # True: the dot fills the blank
print(bool(re.search(r"bill\.pdf", name)))  # False: needs a real dot

# | means or
for w in ["by UPI", "by card", "by cash"]:
    print(w, bool(re.search(r"UPI|card", w)))

# [ ] picks one from the menu
for w in ["cat", "bat", "rat"]:
    print(w, bool(re.search(r"[cb]at", w)))

# ranges: one lowercase letter, one capital, one digit
for w in ["upi", "UPI", "123"]:
    print(w, bool(re.search(r"[a-z]", w)), bool(re.search(r"[A-Z]", w)), bool(re.search(r"[0-9]", w)))

# ^ inside the brackets means not
for w in ["123", "12a"]:
    print(w, bool(re.search(r"[^0-9]", w)))

# short forms: \d a digit, \w a word character, \s whitespace
print(bool(re.search(r"\d", "Room 42")))         # True
for w in ["a", "1", "_", "-"]:
    print(w, bool(re.search(r"\w", w)))
print(bool(re.search(r"\s", "UPI paid")))        # True

# a capital flips the meaning: \D \W \S
print(bool(re.search(r"\D", "123")))             # False
print(bool(re.search(r"\W", "a_1")))             # False
print(bool(re.search(r"\S", "   ")))             # False

# all of it together
print(bool(re.search(r"[aeiou]", "sky")))        # False: no vowel
print(bool(re.search(r"^\w+$", "hello world")))  # False: \w has no space

# + one or more, * zero or more, ? zero or one, of the thing before
for w in ["gd", "god", "good"]:
    print(w, bool(re.search(r"go+d", w)), bool(re.search(r"go*d", w)))
for w in ["color", "colour", "colouur"]:
    print(w, bool(re.search(r"colou?r", w)))

# { } gives the exact count: {5} means exactly five
for number in ["98765-43210", "9876-43210", "98765-432100", "9876543210"]:
    print(number, bool(re.fullmatch(r"\d{5}-\d{5}", number)))

# where these help in real work
print(re.sub(r"[^0-9]", "", "1,25,000"))         # 125000: clean an amount
print(re.findall(r"\d{6}", "Your OTP is 482913"))  # ['482913']: read an OTP
print(re.sub(r"\s+", " ", "UPI    paid   now"))   # UPI paid now: tidy pasted text
print(re.findall(r"\S+@\S+", "Pay sumit@ybl today"))  # ['sumit@ybl']: pull out an ID
print(bool(re.search(r"pdf$", "bill.pdf.exe")))  # False: catch a disguised file

# findall: every match, as a list
sms = "Rs.600 sent to sumit@ybl, Rs.250 to vamsi@oksbi, Rs.1500 to sathvik@paytm"
print(re.findall(r"\d+", sms))
print(re.findall(r"\w+@\w+", sms))


## Regex groups

`06_regex_groups.py`

Capturing groups: brackets in the pattern pull out the pieces you want.


In [ ]:
import re

text = "Name: Rex, Age: 3"
match = re.search(r"Name: (\w+), Age: (\d+)", text)
print(match.group(0))     # the whole match
print(match.group(1))     # Rex
print(match.group(2))     # 3

# findall with groups returns tuples of the groups, not whole matches
text = "Name: Rex, Age: 3; Name: Fido, Age: 5"
print(re.findall(r"Name: (\w+), Age: (\d+)", text))

# a bank SMS: amount, account tail, payee
sms = "Rs.600 debited from a/c XX4321 to SUMIT on 04-11-26 via UPI"
m = re.search(r"Rs\.(\d+) debited from a/c XX(\d{4}) to (\w+)", sms)
amount, tail, payee = m.groups()
print(amount, tail, payee)

# log lines: timestamp, level, message
logs = [
    "2026-11-04 21:05:02 ERROR Failed to connect to database",
    "2026-11-04 21:05:05 INFO Retry succeeded",
    "2026-11-04 21:06:40 WARNING Disk usage above 90%",
]
pattern = r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) (\w+) (.+)"
for line in logs:
    timestamp, level, message = re.match(pattern, line).groups()
    print(f"[{level}] {timestamp} - {message}")


## Paisapay v13

`paisapay_v13.py`

PaisaPay v13: classes from Class 12, finished here, plus regular expressions.
A Transaction and an Account hold their own data, the balance and PIN are private,
a PremiumAccount inherits everything and overrides the fee rule.
New with regex: a real UPI ID check, a PAN check (menu 7), reading a bank SMS (menu 6).


In [ ]:
import re
from functools import reduce

print("Welcome to PaisaPay")

contacts = {"rajani@okaxis": "Rajani", "sumit@ybl": "Sumit"}


def clean(text):
    return text.strip().lower()


def is_valid_upi_id(upi_id):
    # letters, digits or dots, then @, then a bank handle of letters only
    return re.fullmatch(r"[a-z0-9.]+@[a-z]+", upi_id) is not None


def is_valid_pan(pan):
    # five letters, four digits, one letter: ABCDE1234F
    return re.fullmatch(r"[A-Z]{5}\d{4}[A-Z]", pan) is not None


def read_bank_sms(sms):
    # "Rs.600 debited from a/c XX4321 to SUMIT on 04-11-26 via UPI"
    m = re.search(r"Rs\.(\d+) debited from a/c XX(\d{4}) to (\w+)", sms)
    if m is None:
        return None
    amount, tail, payee = m.groups()
    return int(amount), tail, payee.title()


def mode_name(mode):
    match mode:
        case "upi":
            return "UPI"
        case "card":
            return "Card"
        case "netbanking":
            return "Net banking"
        case _:
            return "Unknown mode"


def name_for(upi_id):
    return contacts.get(upi_id, upi_id.split("@")[0].title())


class Transaction:
    def __init__(self, payee, amount, mode, fee, cashback):
        self.payee = payee
        self.amount = amount
        self.mode = mode
        self.fee = fee
        self.cashback = cashback

    def describe(self):
        return f"₹{self.amount} to {self.payee} ({mode_name(self.mode)})"


class Account:
    def __init__(self, owner, balance, pin):
        self.owner = owner
        self.__balance = balance
        self.__pin = pin
        self.history = []

    def get_balance(self):
        return self.__balance

    def check_pin(self, pin):
        return pin == self.__pin

    def fee_for(self, mode):
        if mode == "card":
            return 5
        elif mode == "netbanking":
            return 2
        return 0

    def cashback_for(self, amount):
        if amount >= 5000:
            return 100
        elif amount >= 2000:
            return 50
        elif amount >= 500:
            return 10
        return 0

    def add_money(self, amount):
        if amount <= 0:
            print("Invalid amount. Nothing added.")
            return
        self.__balance += amount
        print(f"Added ₹{amount}. Balance ₹{self.__balance}")

    def pay(self, upi_id, amount, mode="upi"):
        if not is_valid_upi_id(upi_id):
            print(f"'{upi_id}' is not a valid UPI ID. Payment refused.")
            return
        if amount <= 0 or amount > 100000:
            print("Invalid amount. Payment refused.")
            return
        if mode_name(mode) == "Unknown mode":
            print(f"Unknown payment mode '{mode}'. Payment refused.")
            return
        fee = self.fee_for(mode)
        if amount + fee > self.__balance:
            print("Insufficient balance. Payment refused.")
            return
        cashback = self.cashback_for(amount)
        self.__balance = self.__balance - amount - fee + cashback
        payee = name_for(upi_id)
        self.history.append(Transaction(payee, amount, mode, fee, cashback))
        print(f"Paid ₹{amount} to {payee} via {mode_name(mode)}")
        print(f"Fee ₹{fee} | Cashback ₹{cashback} | Balance ₹{self.__balance}")

    def statement(self):
        if len(self.history) == 0:
            print("No payments yet.")
            return
        amounts = list(map(lambda t: t.amount, self.history))
        total = reduce(lambda a, b: a + b, amounts)
        biggest = max(self.history, key=lambda t: t.amount)
        print(f"{self.owner}: {len(self.history)} payments | Spent ₹{total}")
        print(f"Biggest: {biggest.describe()}")
        for t in sorted(self.history, key=lambda t: t.amount):
            print("  " + t.describe())


class PremiumAccount(Account):
    def fee_for(self, mode):
        return 0                      # premium: no fees on any mode

    def cashback_for(self, amount):
        return super().cashback_for(amount) * 2


def ask_pin(account):
    for attempt in range(3):
        pin = input("UPI PIN: ").strip()
        if account.check_pin(pin):
            return True
        print(f"Wrong PIN. {2 - attempt} attempts left")
    return False


kind = clean(input("Account type (basic / premium): "))
if kind == "premium":
    account = PremiumAccount("Rajani", 1000, "4321")
else:
    account = Account("Rajani", 1000, "4321")

choice = ""
while choice != "4":
    print()
    print("1 Pay  2 Balance  3 Statement  4 Exit  5 Add  6 Read SMS  7 Check PAN")
    choice = clean(input("Choose: "))

    if choice == "1":
        upi_id = clean(input("Pay to (UPI ID): "))
        amount = int(input("Amount in rupees: ").strip())
        mode = clean(input("Mode (upi / card / netbanking): "))
        if ask_pin(account):
            account.pay(upi_id, amount, mode)
        else:
            print("Three wrong PINs. Payment refused.")

    elif choice == "2":
        print(f"Balance: ₹{account.get_balance()}")

    elif choice == "3":
        account.statement()

    elif choice == "5":
        account.add_money(int(input("Amount to add: ").strip()))

    elif choice == "6":
        parsed = read_bank_sms(input("Paste the bank SMS: "))
        if parsed is None:
            print("Could not read that SMS.")
        else:
            amount, tail, payee = parsed
            print(f"₹{amount} to {payee} from the account ending {tail}")

    elif choice == "7":
        pan = clean(input("PAN: ")).upper()
        print("Valid PAN" if is_valid_pan(pan) else "Not a valid PAN")

    elif choice != "4":
        print(f"Unknown choice '{choice}'. Try again.")

print("Thank you for using PaisaPay")

# What v13 cannot do yet:
#   - remember anything after the program ends (files and modules, next)


## Your turn

The assignment for this class: https://github.com/prateekScaler/AIML-Jul26-Batch3-Python/blob/main/Class-13-OOP-Practice-And-Regex/ASSIGNMENT.md

Add new cells below and write your own programs here.


In [ ]:
# your code
